In [1]:
# Test core geospatial libraries
import geopandas as gpd
import rasterio
import xarray as xr
import leafmap
import pandas as pd
import numpy as np
from pathlib import Path

# Molly in and out paths
inpath = Path("C:/Users/Molly/OneDrive - University of Tasmania/KGG375_AT3_MM&TP/AT3/Input Data") #all input files
outpath = Path("C:/Users/Molly/OneDrive - University of Tasmania/KGG375_AT3_MM&TP/AT3/Output Data") #all output files

In [2]:
#GITHUB TESTING!!!!
print("hello Molly")
print("TARA!!! :)")

hello Molly
TARA!!! :)


## Clean ABS socio economic factors

In [3]:
g01_input_file = inpath / "2021Census_G01_NSW_LGA.csv"
g02_input_file = inpath / "2021Census_G02_NSW_LGA.csv"
g43_input_file = inpath/ "2021Census_G43_NSW_LGA.csv"

g01_output_file = outpath / "G01_extracted_columns.csv"
g02_output_file = outpath / "G02_extracted_columns.csv"
g43_output_file = outpath / "G43_extracted_columns.csv"

g01_columns = ["LGA_CODE_2021", "Tot_P_P", "High_yr_schl_comp_Yr_12_eq_P", "Age_20_24_yr_P", "Age_25_34_yr_P",
               "Age_35_44_yr_P", "Age_45_54_yr_P", "Age_55_64_yr_P", "Age_65_74_yr_P", "Age_75_84_yr_P", "Age_85ov_P"]
g02_columns = ["LGA_CODE_2021", "Median_tot_hhd_inc_weekly"]
g43_columns = ["LGA_CODE_2021", "lfs_Unmplyed_lookng_for_wrk_P", "lfs_Tot_LF_P"]

socio_economic_data = outpath / "2021Census_NSW_LGA_combined.csv"

def extract_columns(input_path, column_names, output_path):
    """
    """
    csv = pd.read_csv(input_path)
    
    relevant_columns = csv[column_names]
    
    relevant_columns.to_csv(output_path, index=False)
    
    print(f"Data extracted successfully: {output_path}")
    
    return output_path


extract_columns(g01_input_file, g01_columns, g01_output_file)
extract_columns(g02_input_file, g02_columns, g02_output_file)
extract_columns(g43_input_file, g43_columns, g43_output_file)

# Clean g01 data
g01 = pd.read_csv(g01_output_file)
g01["Population_20ov"] = g01["Age_20_24_yr_P"] + g01["Age_25_34_yr_P"] + g01["Age_35_44_yr_P"] + g01["Age_45_54_yr_P"] + g01["Age_55_64_yr_P"] + g01["Age_65_74_yr_P"] + g01["Age_75_84_yr_P"] + g01["Age_85ov_P"]

age_columns = [
    "Age_20_24_yr_P",
    "Age_25_34_yr_P",
    "Age_35_44_yr_P",
    "Age_45_54_yr_P",
    "Age_55_64_yr_P",
    "Age_65_74_yr_P",
    "Age_75_84_yr_P",
    "Age_85ov_P"
]

g01 =g01.drop(columns=age_columns)
g01.to_csv(g01_output_file, index=False)


# Combine all three extracted .csv's into one .csv
g01 = pd.read_csv(g01_output_file)
g02 = pd.read_csv(g02_output_file)
g43 = pd.read_csv(g43_output_file)

combined = (g01.merge(g02, on="LGA_CODE_2021")
            .merge(g43, on="LGA_CODE_2021")
)

combined.to_csv(socio_economic_data, index=False)
print(f"Data merged successfully: {socio_economic_data}")

# Compute basic statistics 
# high school percentage per LGA 
combined["High_School_Completion_Percentage"] = (combined["High_yr_schl_comp_Yr_12_eq_P"] / combined["Population_20ov"]) * 100
print(f"Percentage computed")

# unemplyment percentage per LGA
combined["Unemployment_Percentage"] = (combined["lfs_Unmplyed_lookng_for_wrk_P"] / combined["lfs_Tot_LF_P"]) * 100
print(f"Percentage computed")     

combined.to_csv(socio_economic_data, index=False)

# Aggreate socio-economic to LGA polygons

lga_input_file = inpath / "SydneyLGAs_MGA202056.shp"
socio_economic_data = outpath / "2021Census_NSW_LGA_combined.csv"

lga_socio_economic_output = outpath / "SydneyLGAs_MGA202056_socio_econ.shp"


lga_shp = gpd.read_file(lga_input_file)
census_data = pd.read_csv(socio_economic_data)

lga_shp["LGA_CODE25"] = lga_shp["LGA_CODE25"].astype(str)
census_data["LGA_CODE_2021"] = census_data["LGA_CODE_2021"].astype(str)

lga_merged = lga_shp.merge(census_data, left_on="LGA_CODE25", right_on="LGA_CODE_2021", how="left")

lga_merged.to_file(lga_socio_economic_output)
print(f"LGA boundaries successfully merged with Census data: {lga_socio_economic_output}") 


Data extracted successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\G01_extracted_columns.csv
Data extracted successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\G02_extracted_columns.csv
Data extracted successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\G43_extracted_columns.csv
Data merged successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\2021Census_NSW_LGA_combined.csv
Percentage computed
Percentage computed
LGA boundaries successfully merged with Census data: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\SydneyLGAs_MGA202056_socio_econ.shp


## Hotspot analysis 

In [4]:
# Merge Crime data to LGA shapefile
from pyparsing import col


drug_crime_input_file = outpath / "Drugs_LGA_Totals.csv"
violent_crime_input_file = outpath / "ViolentCrime_LGA_Totals.csv"
property_crime_input_file = outpath / "PropertyOffences_LGA_Totals.csv"
lga_socio_economic_input_file = outpath / "SydneyLGAs_MGA202056_socio_econ.shp"

lga_socioecon_crime_output = outpath / "SydneyLGAs_MGA202056_socioecon_crime_2015_2025.shp"

lga_econ_shp = gpd.read_file(lga_socio_economic_input_file)
drug_data = pd.read_csv(drug_crime_input_file)
property_data = pd.read_csv(property_crime_input_file)
violent_data = pd.read_csv(violent_crime_input_file)

# calculate total crime counts per LGA for each crime type
def crime_count(crime_data, type_name, output_file):
    """
    """
    
    dataframe = crime_data.copy()
    
    year_columns = [col for col in dataframe.columns 
                    if any(str(col).endswith(f"_{year}") for year in range(2015, 2026))
    ]
    
    dataframe[f"{type_name}_Total"] = dataframe[year_columns].sum(axis=1)
    
    dataframe.to_csv(output_file, index=False)
    print(f"Crime counts successfully calculated: {output_file}")
  
    return dataframe


crime_count(drug_data, "Drugs", drug_crime_input_file)
crime_count(property_data, "PO", property_crime_input_file)
crime_count(violent_data, "VC", violent_crime_input_file)



lga_econ_shp["LGA_NAME25"] = lga_econ_shp["LGA_NAME25"].astype(str)
drug_data["LGA"] = drug_data["LGA"].astype(str)
property_data["LGA"] = property_data["LGA"].astype(str)
violent_data["LGA"] = violent_data["LGA"].astype(str)

lga_merged = lga_econ_shp.merge(drug_data, left_on="LGA_NAME25", right_on="LGA", how="left")
lga_merged = lga_merged.merge(property_data, left_on="LGA_NAME25", right_on="LGA", how="left")
lga_merged = lga_merged.merge(violent_data, left_on="LGA_NAME25", right_on="LGA", how="left")

lga_merged.to_file(lga_socioecon_crime_output)
print(f"LGA boundaries successfully merged with Crime data: {lga_socioecon_crime_output}") 


# Calculate crime rates per 1000 people per LGA for each crime type

def crime_rate(lga_data, crime_count_column, population_column, output_column):
    """
    """
    dataframe = lga_data.copy()
    # 2015-2025 = 11 yrs
    years = 11
    
    dataframe[output_column] = ((dataframe[crime_count_column] / dataframe[population_column]) * 1000)
    print(f"Crime rate calculated: {output_column}")
    
    return dataframe
    

lga_merged = crime_rate(lga_merged, "Drugs_Total", "Tot_P_P", "Drugs/1000")  
lga_merged = crime_rate(lga_merged, "PO_Total", "Tot_P_P", "PO/1000")
lga_merged = crime_rate(lga_merged, "VC_Total", "Tot_P_P", "VC/1000")

lga_merged.to_file( outpath / "SydneyLGAs_SpatialCrimeAnalysis.shp" )
print(f"Crime spatial statistics successfully calculated and saved: {outpath / 'SydneyLGAs_SpatialCrimeAnalysis.shp'}")

# # Caluculate hotspots using Getis-Ord Gi* statistic for each crime type
# from esda.getisord import G
# from esda.getisord import G_Local


# complete_lga = gpd.read_file(outpath / "SydneyLGAs_SpatialCrimeAnalysis.shp")


# def getis_ord_analysis(dataframe, crime_column):
#     """
#     """
#     weight = weights.Queen.from_dataframe(complete_lga)
#     weight.transform = "B"

#     y = dataframe[crime_column]
    
#     # global hotspots 
#     # Is there evidence that high or low drug-crime rates are spatially concentrated across the study area?

#     g = G(y, weight)
    
#     print(f"\n--- Global Getis-Ord G: {crime_column} ---")
#     print("Observed G:", g.G)
#     print("Expected G:", g.EG)
#     print("Z-score:", g.z_norm)
#     print("p-value:", g.p_norm)

#     # Local hotpots

#     gi = G_Local(
#         y,
#         weight,
#         star=True,
#         permutations=999,
#         seed=12345
#     )
    
#     dataframe[f"{crime_column}_Gi_Z"] = gi.Zs 
#     dataframe[f"{crime_column}_Gi_P"] = gi.p_sim

#     return dataframe


# def classify_gi(row):

#     if row["Gi_P"] < 0.05:
        
#         if row["Gi_Z"] > 0:
#             return "Hotspot"
        
#         elif row["Gi_Z"] < 0:
#             return "Coldspot"

#     return "Not significant"

# complete_lga["Gi_Category"] = complete_lga.apply(classify_gi, axis=1)




# complete_lga = getis_ord_analysis( complete_lga, "Drug_Rate_1000" ) 
# complete_lga = getis_ord_analysis( complete_lga, "Violent_Rate_1000" ) 
# complete_lga = getis_ord_analysis( complete_lga, "Property_Rate_1000" )

Crime counts successfully calculated: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\Drugs_LGA_Totals.csv
Crime counts successfully calculated: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\PropertyOffences_LGA_Totals.csv
Crime counts successfully calculated: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\ViolentCrime_LGA_Totals.csv
LGA boundaries successfully merged with Crime data: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\SydneyLGAs_MGA202056_socioecon_crime_2015_2025.shp
Crime rate calculated: Drugs/1000
Crime rate calculated: PO/1000
Crime rate calculated: VC/1000
Crime spatial statistics successfully calculated and saved: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\SydneyLGAs_SpatialCrimeAnalysis.shp


In [9]:
from esda.getisord import G
from esda.getisord import G_Local
from libpysal import weights


from libpysal import weights
from esda.getisord import G_Local

complete_lga = gpd.read_file(
    outpath / "SydneyLGAs_SpatialCrimeAnalysis.shp"
)

weight = weights.Queen.from_dataframe(
    complete_lga,
    use_index=False
)

weight.transform = "B"

y = complete_lga["Drugs/1000"].values

gi = G_Local(
    y,
    weight,
    star=True,
    permutations=99
)

print(gi.Zs)



[ 0.83624927  0.00335168 -0.15810808  0.10049471  0.02773971 -0.10943365
  0.03911422  0.09419624  0.17125232 -0.11795089 -0.50616176 -0.58039915
  0.69457148 -0.63188699 -0.44174167  0.06673991 -0.36762296 -0.51035264
 -0.69527536 -0.3007901   0.17724707  1.24150813 -0.47200615 -0.06688739
  0.11798137  1.0590187  -0.30582626  0.48837254 -0.5575682   1.57423824]


In [18]:
# Calculate hotspots using Getis-Ord Gi* statistic for each crime type
from esda.getisord import G
from esda.getisord import G_Local
from libpysal import weights


complete_lga = gpd.read_file(
    outpath / "SydneyLGAs_SpatialCrimeAnalysis.shp"
)


def getis_ord_analysis(dataframe, crime_column, crime_name):

    weight = weights.Queen.from_dataframe(
        dataframe,
        use_index=False
    )

    weight.transform = "B"

    y = dataframe[crime_column]

    # Global Getis-Ord G
    g = G(y, weight)

    print(f"\n--- Global Getis-Ord G: {crime_name} ---")
    print("Observed G:", g.G)
    print("Expected G:", g.EG)
    print("Z-score:", g.z_norm)
    print("p-value:", g.p_norm)

    # Local Getis-Ord Gi*
    gi = G_Local(
        y,
        weight,
        star=True,
        permutations=999,
        seed=12345,
        n_jobs=1
    )

    dataframe[f"{crime_name}_Z"] = gi.Zs
    dataframe[f"{crime_name}_P"] = gi.p_sim

    return dataframe


def classify_gi(dataframe, crime_column):
    """
    Classifies LGAs as Hotspot, Coldspot or Not significant
    based on the Local Getis-Ord Gi* results.
    """

    def classify(row):

        z_score = row[f"{crime_column}_Z"]
        p_value = row[f"{crime_column}_P"]

        if p_value < 0.05:

            if z_score > 0:
                return "Hotspot"

            elif z_score < 0:
                return "Coldspot"

        return "Not significant"

    dataframe[f"{crime_column}_Gi_Category"] = dataframe.apply(
        classify,
        axis=1
    )

    return dataframe

# Drug crime
complete_lga = getis_ord_analysis(
    complete_lga,
    "Drugs/1000",
    "Drug"
)

complete_lga = classify_gi(
    complete_lga,
    "Drug"
)

# Violent Crime
complete_lga = getis_ord_analysis(
    complete_lga,
    "VC/1000",
    "VC"
)

complete_lga = classify_gi(
    complete_lga,
    "VC"
)

# Property Offences
complete_lga = getis_ord_analysis(
    complete_lga,
    "PO/1000",
    "PO"
)

complete_lga = classify_gi(
    complete_lga,
    "PO"
)


complete_lga[
    [
        "LGA",
        "Drug_Z",
        "Drug_P",
        "Drug_Gi_Category",
        "VC_Z",
        "VC_P",
        "VC_Gi_Category",
        "PO_Z",
        "PO_P",
        "PO_Gi_Category"
    ]
]

complete_lga.to_file(
    outpath / "SydneyLGAs_GetisOrd.shp"
)

: 